# 프로젝트 Notebook 01. pandas로 상권 데이터 이해하기

목표: CSV를 읽고 관측 단위·자료형·품질을 확인한 뒤 필터, 파생변수, 집계와 기본 시각화를 수행한다.
Course 02의 lecture_note.md를 먼저 읽고 이 Notebook을 실행한다.

In [ ]:
from pathlib import Path
import sys, subprocess

REPO_URL = "https://github.com/niko2204/bigdataservice.git"
if "google.colab" in sys.modules:
    ROOT = Path("/content/bigdataservice")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
else:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    ROOT = next((p.resolve() for p in candidates if (p / "data").exists()), None)
    if ROOT is None:
        raise FileNotFoundError("bigdataservice 저장소 안에서 실행하세요.")
print("저장소:", ROOT)

## 1. 읽기와 관측 단위

location_features는 행정동 한 행, stores는 개별 점포 한 행이다. 관측 단위가 다른 표를 바로 병합하면 행이 늘어날 수 있다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

locations = pd.read_csv(ROOT / "data/sample/location_features.csv")
stores = pd.read_csv(ROOT / "data/sample/stores.csv")
print("지역:", locations.shape, "점포:", stores.shape)
display(locations.head(3))
display(stores.head(3))

## 2. 구조와 품질 프로파일

자료형, 결측, 고유값 수와 식별자 중복을 확인한다. 열 이름만 보고 단위를 추측하지 말고 데이터 사전과 대조한다.

In [ ]:
profile = pd.DataFrame({
    "자료형": locations.dtypes.astype(str),
    "결측수": locations.isna().sum(),
    "고유값수": locations.nunique(dropna=False),
})
display(profile)
print("행정동명 중복:", locations["행정동명"].duplicated().sum())
print("점포번호 중복:", stores["상가업소번호"].duplicated().sum())
assert locations["행정동명"].is_unique

## 3. 조건 필터와 정렬

Series 조건에는 and 대신 &, or 대신 |를 쓰고 각 조건을 괄호로 묶는다.

In [ ]:
condition = (locations["20대인구"] >= 1500) & (locations["카페수"] <= 30)
candidates = (
    locations.loc[condition, ["행정동명", "20대인구", "유동인구", "카페수"]]
             .sort_values("유동인구", ascending=False)
)
display(candidates)

## 4. 파생변수와 0으로 나누기

점포가 0개일 때 분모에 무조건 1을 더하면 지표 의미가 달라진다. 여기서는 계산 불가인 NaN으로 둔다.

In [ ]:
locations["카페당20대인구"] = np.where(
    locations["카페수"] > 0,
    locations["20대인구"] / locations["카페수"],
    np.nan,
)
display(locations[["행정동명", "카페당20대인구"]].sort_values("카페당20대인구", ascending=False).head())

## 5. 그룹 집계

점포 식별자의 nunique를 사용하여 중복에 안전하게 집계한다.

In [ ]:
store_summary = (
    stores.groupby(["행정동명", "상권업종대분류명"], as_index=False)
          .agg(점포수=("상가업소번호", "nunique"),
               세부업종수=("상권업종소분류명", "nunique"))
)
display(store_summary)

## 6. 시각화와 세 문장 해석

In [ ]:
ordered = locations.sort_values("유동인구")
plt.figure(figsize=(8, 5))
plt.barh(ordered["행정동명"], ordered["유동인구"])
plt.xlabel("유동인구 지수"); plt.ylabel("행정동")
plt.title("목포시 교육용 행정동별 유동인구")
plt.tight_layout(); plt.show()

관찰에는 그래프에서 직접 읽은 값, 해석에는 가능한 의미, 한계에는 데이터로 말할 수 없는 것을 쓴다.

## 7. 독립 연습

1. 음식점당 총인구와 편의점당 총인구를 계산한다.
2. 세 업종의 점포당 인구 상위 5개를 한 표로 비교한다.
3. 유동인구 상위와 점포당 인구 상위가 다른 이유를 5문장으로 설명한다.
4. 행 수와 값 범위를 확인하는 assert를 추가한다.

In [ ]:
# TODO: 세 업종 비교표
comparison = None
display(comparison)